# Start here: shared IPI and platform

These cells use **recorded hardware evidence** by default. They do not program a board.
Use the configured Python (SCU35 PMC) kernel. Fresh hardware commands are shown in Markdown and must be invoked explicitly after configuring your exact SCU35 and recovery image.
Current PL timing uses AXI Timer; native PMC timing uses `mcycle`. JTAG/UART are outside operation timing.

Follow IPI → XSA → BSP → application ELF → PDI. The PL MicroBlaze V runs the application; the dedicated PMC processor runs PLM. No eFUSE/flash writes.

From the lab directory: `python lab.py doctor`, then `python lab.py build`. Inspect `hw/sys.bd`, `hw/sec_top.xdc`, `<build_dir>/hw.xsa` and the generated BSP before running hardware.

In [1]:
from pathlib import Path
import hashlib, json, sys, time
import pandas as pd
# Works from the notebook directory, lab root, or parent repository.
candidates = [Path.cwd(), *Path.cwd().parents]
ROOT = next((p for base in candidates for p in (base, base / "scu35") if (p / "lab.py").is_file()), None)
assert ROOT is not None, "Launch Jupyter inside this checkout or its scu35 directory"
sys.path.insert(0, str(ROOT)); sys.path.insert(0, str(ROOT / "host"))
from compare import parse_capture
from lab import config
capture = parse_capture(ROOT / "evidence/timer/uart.txt", "7d1bf77bc454dd1f")
rows = pd.DataFrame(capture["rows"])
print("Recorded board capture: 31 complete rows; current Python:", sys.version.split()[0])


Recorded board capture: 31 complete rows; current Python: 3.11.9


In [2]:
print("Root:", ROOT)
print("Build directory:", config()["build_dir"])
print("Firmware ID:", capture["metadata"]["build_id"])
display(rows[["algorithm", "case", "size_bytes", "scu35_operation_mean_us"]])

Root: C:\aiup\amdpub\scu35
Build directory: C:\aiup\amdpub\scu35\out
Firmware ID: 7d1bf77bc454dd1f


,algorithm,case,size_bytes,scu35_operation_mean_us
0,SHA3-256,valid,16,23.7205
1,SHA3-256,valid,64,23.7235
2,SHA3-256,valid,256,23.9360
3,SHA3-256,valid,1024,25.5150
4,SHA3-256,valid,2048,27.6170
5,SHAKE-256,valid,16,23.7460
6,SHAKE-256,valid,64,23.7465
7,SHAKE-256,valid,256,23.9545
8,SHAKE-256,valid,1024,25.5325
9,SHAKE-256,valid,2048,27.6390


## Fresh hardware: explicit step

Configure `config.json` first. Run `python lab.py hardware --pdi /path/to/build/demo.pdi` from the lab directory for the full runtime suite, or the native/boot command above for that path. Save logs, verify recovery and compare results before discussing speed. See [audit checkpoints](../docs/audit.md).